# **Demo 02: Executing Serial and Parallel Workflows Using CrewAI Agents**


**Objective:** To demonstrate how to build intelligent workflows using CrewAI agents that execute tasks either sequentially (serial) or independently (parallel). This demo highlights how tasks can be structured based on interdependence or concurrency.

**Prerequisites:** Anthropic API key, Tavily key

**Tools required:** Python (Google Colab)

In [ ]:
# Step 1: Install the required libraries
!pip install -q "crewai[anthropic]" langchain langchain-community langchain-tavily tavily-python pydantic litellm nest_asyncio

In [ ]:
import requests
import nest_asyncio
from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai.tools import BaseTool

# Colab's event loop needs this patch for CrewAI's async kickoff
nest_asyncio.apply()

In [ ]:
# Step 2: Load API keys from Colab Secrets (the key icon in the left sidebar)
# Add ANTHROPIC_API_KEY and TAVILY_API_KEY there before running this cell
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
# --------------------------------------------------
# Tavily Web Search Tool
# --------------------------------------------------
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": os.environ["TAVILY_API_KEY"],
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# --------------------------------------------------
# Claude LLM via CrewAI's native LiteLLM bridge
# Swap the model string for whichever Claude model your API access supports
# --------------------------------------------------
llm = LLM(
    model="anthropic/claude-sonnet-4-5-20250929",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=1500,
    is_litellm=True
)

In [ ]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=3,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 100 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = await crew_serial.kickoff_async()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)

In [ ]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 100 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = await crew_parallel.kickoff_async()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)